# Phase 6B — Multiclass check (RQ5)
**Question:** do the binary findings still hold when there are 3 or more classes?

**Setup** (run by `src/multiclass.py`, resumable like the other phases):
- **Datasets:** all OpenML-CC18 multiclass datasets with ≤ 50,000 rows, ≤ 100 features, no missing values and smallest class ≥ 5 %. That gives **20 datasets** with 3–11 classes (`results/tables/datasets_multiclass.csv`).
- **Noise:** symmetric (a flipped label moves to a random *other* class); **pair-flip** (class k → k+1, the standard multiclass version of class-conditional noise); boundary (flip to the reference model's most likely *other* class, chance ∝ 1 − top-class probability)
- Rates 10, 20, 30 %; the same 6 models; R = 51; clean test labels

**Decomposition with K classes:** where the main prediction is wrong, a disagreeing model only *lowers* the error if it predicts the **true** class (with 3+ classes it may pick another wrong class). So the helpful part of the biased variance is P(prediction = true class). The identity error = bias + unbiased variance − helpful biased variance holds exactly, and reduces to the binary formula when K = 2 (tested in `src/multiclass.py`).

Kernel: **Python (labelnoise)**.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare

NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
sys.path.insert(0, str(BASE / "src"))
import multiclass
multiclass.run_all()                      # resumable: finishes instantly if everything is saved

TABLES = BASE / "results" / "tables"
M = ["Tree", "Bagging", "RF", "AdaBoost", "GB", "XGB"]
T = ["symmetric", "pair_flip", "boundary"]
FAM = {"Tree": "single", "Bagging": "parallel", "RF": "parallel", "AdaBoost": "sequential", "GB": "sequential", "XGB": "sequential"}

d = pd.concat([pd.read_csv(f) for f in sorted((BASE / "results" / "raw" / "rq5").glob("*.csv"))], ignore_index=True)
ok = len(d) == 1200 and not d.isna().any().any() and d.identity_gap.abs().max() < 1e-12 \
     and not d.duplicated(["dataset", "model", "noise_type", "rho"]).any()
print(f"{'OK  ' if ok else 'FAIL'}  {len(d)} rows, {d.dataset.nunique()} datasets, {d.classes.min()}-{d.classes.max()} classes")
pd.read_csv(TABLES / "datasets_multiclass.csv")

19:35:06  START rq5: 20 datasets x 6 models x 10 conditions, R = 51


19:35:10  DONE rq5


OK    1200 rows, 20 datasets, 3-11 classes


,dataset,openml_id,rows,classes,features_after_encoding,smallest_class_share,train_rows,test_rows
0,balance-scale,11,625,3,4,0.078,437,188
1,analcatdata_dmft,469,797,6,21,0.154,557,240
2,analcatdata_authorship,458,841,4,70,0.065,588,253
3,vehicle,54,846,4,18,0.235,592,254
4,vowel,307,990,11,27,0.091,693,297
5,cmc,23,1473,3,24,0.226,1031,442
6,mfeat-karhunen,16,2000,10,64,0.100,1400,600
7,mfeat-fourier,14,2000,10,76,0.100,1400,600
8,mfeat-zernike,22,2000,10,47,0.100,1400,600
9,mfeat-morphological,18,2000,10,6,0.100,1400,600


## Damage relative to clean labels
As in the binary study: Δ = value under noise − value on clean labels, for the same dataset and model. Clean error is shown too, because a model that is already poor on clean data can behave oddly under noise.

In [2]:
c = d[d.noise_type == "none"].set_index(["dataset", "model"])
n = d[d.noise_type != "none"].copy()
k = list(zip(n.dataset, n.model))
for col in ["error", "bias", "net_var"]:
    n["d_" + col] = n[col].to_numpy() - c.loc[k, col].to_numpy()
n["family"] = n.model.map(FAM)
print("Clean error (%) by model:", (c.groupby("model").error.mean() * 100).reindex(M).round(1).to_dict())

Clean error (%) by model: {'Tree': 27.3, 'Bagging': 21.4, 'RF': 19.7, 'AdaBoost': 33.2, 'GB': 20.8, 'XGB': 20.2}


## RQ5a — Bias or variance? (20 % noise)
Compare with the binary RQ2 table: symmetric ≈ all variance? boundary ≈ split? class-conditional?

In [3]:
g = n[np.isclose(n.rho, 0.2)].groupby(["noise_type", "model"])[["d_error", "d_bias", "d_net_var"]].mean()
g["variance share %"] = 100 * g.d_net_var / g.d_error
g[["d_error", "d_bias", "d_net_var"]] *= 100
g = g.rename(columns={"d_error": "Δ error (pp)", "d_bias": "Δ bias (pp)", "d_net_var": "Δ net var (pp)"})
g = g.reindex(pd.MultiIndex.from_product([T, M], names=["noise_type", "model"])).round(1)
g.to_csv(TABLES / "phase6b_split_20pct.csv")
g

Δ error (pp)  Δ bias (pp)  Δ net var (pp)  \
noise_type model                                                 
symmetric  Tree              13.0         -0.4            13.4   
           Bagging            1.8         -0.6             2.4   
           RF                 1.9         -0.1             2.0   
           AdaBoost          -2.2         -3.5             1.3   
           GB                 3.3         -0.2             3.5   
           XGB                2.2         -0.3             2.5   
pair_flip  Tree              10.9         -0.4            11.3   
           Bagging            3.5          0.0             3.4   
           RF                 3.3          0.2             3.1   
           AdaBoost           8.0          7.6             0.4   
           GB                 5.2          0.9             4.3   
           XGB                4.3          1.1             3.2   
boundary   Tree               8.4          2.5             5.9   
           Bagging            4.1          1.7             2.4   
           RF                 4.2          1.9             2.3   
           AdaBoost           1.8          0.8             1.0   
           GB                 4.1          2.0             2.1   
           XGB                3.7          1.8             1.9   

                     variance share %  
noise_type model                       
symmetric  Tree                 103.0  
           Bagging              130.9  
           RF                   103.9  
           AdaBoost             -57.6  
           GB                   106.0  
           XGB                  114.0  
pair_flip  Tree                 103.3  
           Bagging               98.6  
           RF                    93.9  
           AdaBoost               5.0  
           GB                    82.8  
           XGB                   74.9  
boundary   Tree                  70.3  
           Bagging               59.2  
           RF                    54.6  
           AdaBoost              56.5  
           GB                    51.7  
           XGB                   51.4

In [4]:
f = n.groupby(["noise_type", "family", "rho"])[["d_error", "d_net_var"]].mean()
share = (100 * f.d_net_var / f.d_error).unstack("rho").round(0)
share.columns = [f"{int(r * 100)} %" for r in share.columns]
share.to_csv(TABLES / "phase6b_variance_share_by_family.csv")
print("Variance share (%) by noise type, family and rate")
share

Variance share (%) by noise type, family and rate


10 %   20 %   30 %
noise_type family                          
boundary   parallel      83.0   57.0   37.0
           sequential    75.0   52.0   36.0
           single       100.0   70.0   50.0
pair_flip  parallel     113.0   96.0   91.0
           sequential    58.0   45.0   44.0
           single       102.0  103.0   95.0
symmetric  parallel     131.0  117.0  109.0
           sequential  1133.0  224.0  148.0
           single       103.0  103.0  100.0

## RQ5b — Which models are most robust? (Friedman test on damage ranks, 20 % noise)

In [5]:
rows = {}
for nt in T:
    w = n[(n.noise_type == nt) & np.isclose(n.rho, 0.2)].pivot(index="dataset", columns="model", values="d_error")[M]
    rows[nt] = {**w.rank(axis=1).mean().round(2).to_dict(), "Friedman p": friedmanchisquare(*[w[m] for m in M]).pvalue}
ranks = pd.DataFrame(rows)
ranks.to_csv(TABLES / "phase6b_ranks.csv")
ranks

,symmetric,pair_flip,boundary
Tree,5.750000e+00,5.400000e+00,5.500000
Bagging,2.750000e+00,2.620000e+00,3.300000
RF,2.950000e+00,2.220000e+00,3.500000
AdaBoost,2.000000e+00,3.800000e+00,2.150000
GB,4.350000e+00,4.000000e+00,3.580000
XGB,3.200000e+00,2.950000e+00,2.980000
Friedman p,7.258452e-10,3.740919e-07,0.000001


## Findings (compared with the binary study)
- **Symmetric noise → variance** (≈ 100 % for every model with a reasonable clean error): the binary finding H1 holds.
- **Boundary noise → split, with bias growing as the rate rises**: the binary finding H3 holds.
- **Class-conditional (pair-flip) noise:** stump-based AdaBoost is almost pure bias, as in binary; GB and XGB are mostly variance. This fits the binary capacity result (class-conditional noise gives bias in low-capacity models and variance in high-capacity ones). Unlike binary one-sided noise, the models differ significantly here, and **bagging and random forests are the most robust**.
- **Caveat:** AdaBoost with stumps fits multiclass data poorly (clean error ≈ 33 % vs ≈ 20 % for the other ensembles). Its error even falls slightly under symmetric noise, so its multiclass numbers should be read with care.